In [0]:
# 10_rag_evidence_seed — Gold Risk Case -> RAG evidence document/chunk
# RAG 품질, prompt, embedding 설정은 DS/AI 영역이다.
# 이 notebook은 RAG가 참조할 근거 document/chunk 테이블만 생성한다.

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("run_id", "manual")

CATALOG = dbutils.widgets.get("catalog")
RUN_ID = dbutils.widgets.get("run_id")

In [0]:
gold_count = spark.table(f"{CATALOG}.gold.gold_core").count()
if gold_count == 0:
    raise RuntimeError("gold.gold_core has no rows. Run intermediate_task and gold_task before rag_evidence_task.")

print(f"gold_core: {gold_count} rows")

In [0]:
documents = spark.sql(f"""
SELECT
  g.risk_case_key,
  g.finding_id,
  CONCAT('Access Drift Risk Case: ', g.subject_display_name, ' -> ', g.asset_display_name) AS document_title,
  CONCAT_WS('\n',
    CONCAT('Risk Case Key: ', g.risk_case_key),
    CONCAT('Drift Type: ', g.drift_type),
    CONCAT('Subject: ', g.subject_display_name, ' (', g.subject_type, ', source_id=', g.subject_source_principal_id, ')'),
    CONCAT('Owner: ', g.owner_display_name),
    CONCAT('Asset: ', g.asset_display_name, ' (source_id=', g.asset_source_id, ', environment=', g.asset_environment, ', sensitivity=', g.asset_sensitivity, ')'),
    CONCAT('Severity: ', g.severity),
    CONCAT('Evidence Summary: ', g.evidence_summary),
    CONCAT('Recommended Action: ', g.recommended_action),
    CONCAT('Data Scope: ', COALESCE(ac.data_scope_label, 'unknown')),
    CONCAT('Sensitive Data Examples: ', COALESCE(array_join(ac.sensitive_data_examples, ', '), 'unknown')),
    CONCAT('Sensitivity Reason: ', COALESCE(ac.sensitivity_reason, 'unknown')),
    CONCAT('Assignment: ', COALESCE(ar.assignee_team, 'unassigned'), ' / ', COALESCE(ar.assignee_name, 'unassigned')),
    CONCAT('Action State: ', COALESCE(s.status, 'open')),
    'This evidence is for SecOps review support. It must not be treated as an automatic revoke instruction.'
  ) AS document_text,
  array('gold.gold_core', 'context.asset_context', 'context.assignment_routing', 'app.risk_action_state') AS source_table_refs,
  map(
    'risk_case_key', g.risk_case_key,
    'finding_id', g.finding_id,
    'subject_principal_id', g.subject_principal_id,
    'subject_source_principal_id', g.subject_source_principal_id,
    'owner_principal_id', g.owner_principal_id,
    'asset_id', g.asset_id,
    'asset_source_id', g.asset_source_id,
    'credential_id', g.credential_id,
    'owner_edge_id', g.owner_edge_id,
    'access_edge_id', g.access_edge_id
  ) AS evidence_keys,
  current_timestamp() AS generated_at,
  '{RUN_ID}' AS run_id
FROM {CATALOG}.gold.gold_core g
LEFT JOIN {CATALOG}.context.asset_context ac
  ON g.asset_source_id = ac.source_asset_id
LEFT JOIN {CATALOG}.context.assignment_routing ar
  ON g.subject_type = ar.subject_type
 AND g.subject_source_principal_id = ar.subject_id
 AND g.asset_source_id = ar.asset_id
 AND g.drift_type = ar.drift_type
LEFT JOIN {CATALOG}.app.risk_action_state s
  ON g.risk_case_key = s.risk_case_key
""")

(documents.write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.rag.risk_case_documents"))

document_count = spark.table(f"{CATALOG}.rag.risk_case_documents").count()
assert document_count > 0, "risk_case_documents has no rows"
print(f"risk_case_documents: {document_count} rows")

In [0]:
chunks = spark.sql(f"""
SELECT
  sha2(CONCAT('rag_chunk:', risk_case_key, ':0'), 256) AS chunk_id,
  risk_case_key,
  document_text AS chunk_text,
  sha2(document_text, 256) AS chunk_hash,
  CAST(0 AS INT) AS chunk_order,
  document_title AS source_title,
  CONCAT('uc://', '{CATALOG}', '.rag.risk_case_documents/', risk_case_key) AS source_uri,
  source_table_refs,
  evidence_keys,
  generated_at
FROM {CATALOG}.rag.risk_case_documents
""")

(chunks.write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.rag.knowledge_chunks"))

chunk_count = spark.table(f"{CATALOG}.rag.knowledge_chunks").count()
duplicate_chunks = spark.sql(f"""
  SELECT COUNT(*) - COUNT(DISTINCT chunk_id) AS duplicate_count
  FROM {CATALOG}.rag.knowledge_chunks
""").first()["duplicate_count"]
assert chunk_count > 0, "knowledge_chunks has no rows"
assert duplicate_chunks == 0, f"duplicate chunk_id count: {duplicate_chunks}"
print(f"knowledge_chunks: {chunk_count} rows")
print(f"duplicate_chunk_id: {duplicate_chunks}")

In [0]:
forbidden_pattern = "(?i)(secret value|token value|private key value|BEGIN PRIVATE KEY)"
forbidden_documents = spark.sql(f"""
  SELECT COUNT(*) AS cnt
  FROM {CATALOG}.rag.risk_case_documents
  WHERE document_text RLIKE '{forbidden_pattern}'
""").first()["cnt"]
forbidden_chunks = spark.sql(f"""
  SELECT COUNT(*) AS cnt
  FROM {CATALOG}.rag.knowledge_chunks
  WHERE chunk_text RLIKE '{forbidden_pattern}'
""").first()["cnt"]

assert forbidden_documents == 0, f"forbidden value pattern in risk_case_documents: {forbidden_documents}"
assert forbidden_chunks == 0, f"forbidden value pattern in knowledge_chunks: {forbidden_chunks}"

spark.table(f"{CATALOG}.rag.knowledge_chunks").select(
    "chunk_id",
    "risk_case_key",
    "source_title",
    "source_uri",
).show(truncate=False)

summary = [
    "rag evidence seed smoke test passed",
    f"risk_case_documents: {document_count} rows",
    f"knowledge_chunks: {chunk_count} rows",
    f"duplicate_chunk_id: {duplicate_chunks}",
    f"forbidden_documents: {forbidden_documents}",
    f"forbidden_chunks: {forbidden_chunks}",
]
dbutils.notebook.exit("\n".join(summary))